In [1]:
# Empty CustomerIDs should be removed.
# Negative Quantities should be removed.
# Zero and negative UnitPrices should be removed.
# The Sales column should be created.


In [2]:
#-------------------------مرحله 1:-------------- شناخت داده
# InvoiceNo : شماره فاکتور یا سفارش
# StockCode : کد محصول در انبار
# Description : نام محصول
import pandas as pd

df = pd.read_csv(r"online_retail.csv")
print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [3]:
print(df.shape)

(541909, 8)


In [4]:
# 541909 رکورد داریم
# 8 ستون داریم
# ستون CustomerID داده گمشده دارد
# ستون Description داده گمشده دارد
# InvoiceDate هنوز تاریخ نیست و متن است
# Quantity عدد صحیح است
# UnitPrice عدد اعشاری است

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB
None


In [5]:
#----------------تبدیل متن به تاریخ-----------------------

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])


In [6]:
# mean : به طور میانگین در هر ردیف(فاکتور) چند محصول ثبت شده است.
# df.groupby("CustomerID")["Quantity"].sum().mean() میانگین تعداد کالا در هر مشتری
#df.groupby("InvoiceNo")["Quantity"].sum().mean() میانگین تعداد کالا در هر سفارش
# min : نشانه مرجوعی یا لغو./فروش منفی
# df.describe().docxتوصیحات کامل در فایل ورد 

df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [7]:
#----------مهم‌ترین ابزارهای بررسی کیفیت داده------------
print(df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [8]:
#-----------------------پاکسازی داده---------------------------
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df = df.dropna(subset=["CustomerID"])

df = df[df["Quantity"] > 0]

df = df[df["UnitPrice"] > 0]

df["Sales"] = df["Quantity"] * df["UnitPrice"]

In [9]:
# ----------------با توجه به انجام پاک سازی باید مقادیر 0 شوند------------------
print(df.shape)

print(df["CustomerID"].isnull().sum())

print((df["Quantity"] < 0).sum())

print((df["UnitPrice"] <= 0).sum())

(397884, 9)
0
0
0


In [10]:
#----------------------KPIهای کسب‌وکار--------------

print("Total Sales:", df["Sales"].sum()) #مجموع فروش کل فروشگاه

print("Customers:", df["CustomerID"].nunique()) #تعداد مشتریان منحصربه‌فرد

print("Orders:", df["InvoiceNo"].nunique()) #تعداد سفارش‌های ثبت‌شده

Total Sales: 8911407.904
Customers: 4338
Orders: 18532


In [11]:
#------------هر سفارش به طور متوسط چقدر درآمد ایجاد کرده است؟--------

aov = df["Sales"].sum() / df["InvoiceNo"].nunique()

print(aov)

480.8659563997409


In [12]:
#---------------- پرفروش‌ترین محصولات------------------
# POSTAGE,Manual اینا کالا نیستند

top_products = (
    df.groupby("Description")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(top_products)

Description
PAPER CRAFT , LITTLE BIRDIE           168469.60
REGENCY CAKESTAND 3 TIER              142592.95
WHITE HANGING HEART T-LIGHT HOLDER    100448.15
JUMBO BAG RED RETROSPOT                85220.78
MEDIUM CERAMIC TOP STORAGE JAR         81416.73
POSTAGE                                77803.96
PARTY BUNTING                          68844.33
ASSORTED COLOUR BIRD ORNAMENT          56580.34
Manual                                 53779.93
RABBIT NIGHT LIGHT                     51346.20
Name: Sales, dtype: float64


In [13]:
# ---- زمانی که به عددی در نتایج مشکوک می شویم آیا درست هست یا داده پرت؟؟----
df[df["Description"] == "PAPER CRAFT , LITTLE BIRDIE"] \
    .sort_values("Sales", ascending=False)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom,168469.6


In [14]:
#-------------بهترین مشتری فروشگاه--------------
best_customer = (
    df.groupby("CustomerID")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(best_customer)

CustomerID
14646.0    280206.02
18102.0    259657.30
17450.0    194550.79
16446.0    168472.50
14911.0    143825.06
12415.0    124914.53
14156.0    117379.63
17511.0     91062.38
16029.0     81024.84
12346.0     77183.60
Name: Sales, dtype: float64


In [15]:
#-------------متوسط فروش هر مشتری--------------
# 8911407 / 4338 ≈ 2054
df["Sales"].sum() / df["CustomerID"].nunique()

np.float64(2054.2664601198708)

In [16]:
#-----------بهترین ماه فروش-----------------------
#تبدیل متن به دیت
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
# اضافه کردن ستون ماه
df["Month"] = df["InvoiceDate"].dt.to_period("M")

monthly_sales = (
    df.groupby("Month")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

print(monthly_sales)


Month
2011-11    1161817.380
2011-10    1039318.790
2011-09     952838.382
2011-05     678594.560
2011-06     661213.690
2011-08     645343.900
2011-07     600091.011
2011-03     595500.760
2010-12     572713.890
2011-01     569445.040
2011-12     518192.790
2011-04     469200.361
2011-02     447137.350
Freq: M, Name: Sales, dtype: float64


In [17]:
# خلاصه پروژه
# | KPI               |                         مقدار |
# | ----------------- | ------------------------:   |
# | کل فروش        |                  8,911,408    |
# | تعداد مشتری       |                       4,338 |
# | تعداد سفارش       |                      18,532 |
# | ر  بهترین ماه فروش   |                 نوامبر 2011 |
# | فروش بهترین ماه   |                   1,161,817 |
# | پرفروش‌ترین محصول | PAPER CRAFT , LITTLE BIRDIE |
# | بهترین مشتری      |             Customer 14646 |


In [18]:
#--------------پرفروش‌ترین محصول از نظر تعداد-----------
top_quantity = (
    df.groupby("Description")["Quantity"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(top_quantity)

Description
PAPER CRAFT , LITTLE BIRDIE           80995
MEDIUM CERAMIC TOP STORAGE JAR        77916
WORLD WAR 2 GLIDERS ASSTD DESIGNS     54415
JUMBO BAG RED RETROSPOT               46181
WHITE HANGING HEART T-LIGHT HOLDER    36725
ASSORTED COLOUR BIRD ORNAMENT         35362
PACK OF 72 RETROSPOT CAKE CASES       33693
POPCORN HOLDER                        30931
RABBIT NIGHT LIGHT                    27202
MINI PAINT SET VINTAGE                26076
Name: Quantity, dtype: int64


In [19]:
#-------------قیمت متوسط هر محصول------------------
# موارد زیر کالا نیستند بنابراین باید حذف شوند
    # "AMAZON FEE",
    # "CRUK Commission",
    # "Manual",
    # "DOTCOM POSTAGE",
    # "Bank Charges"

df.groupby("Description")["UnitPrice"] \
  .mean() \
  .sort_values(ascending=False) \
  .head(10)

Description
DOTCOM POSTAGE                        744.147500
PICNIC BASKET WICKER 60 PIECES        649.500000
Manual                                175.291585
RUSTIC  SEVENTEEN DRAWER SIDEBOARD    158.076923
REGENCY MIRROR WITH SHUTTERS          156.428571
VINTAGE BLUE KITCHEN CABINET          146.750000
VINTAGE RED KITCHEN CABINET           143.421053
CHEST NATURAL WOOD 20 DRAWERS         118.076923
LOVE SEAT ANTIQUE WHITE METAL         114.024390
VINTAGE POST OFFICE CABINET            66.360000
Name: UnitPrice, dtype: float64

In [20]:
#------------حذف چند مقدار مشخص---------------
remove_items = [
    "AMAZON FEE",
    "CRUK Commission",
    "Manual",
    "DOTCOM POSTAGE",
    "Bank Charges",
    "Discount"
]

df = df[~df["Description"].isin(remove_items)]


In [21]:
df["Description"].isin(remove_items)

0         False
1         False
2         False
3         False
4         False
          ...  
541904    False
541905    False
541906    False
541907    False
541908    False
Name: Description, Length: 397572, dtype: bool

In [22]:
print(df["Description"].isin(remove_items).sum())

0


In [23]:
#------------تحلیل کشورها-------------
#----------درآمد از کدام کشور می آید؟------------
# 7276121 / 8911408 * 100 سهم uk (81.6%)
country_sales = (
    df.groupby("Country")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

print(country_sales.head(10))

Country
United Kingdom    7276121.253
Netherlands        285446.340
EIRE               262171.560
Germany            226570.890
France             199531.680
Australia          138521.310
Spain               61577.110
Switzerland         56443.950
Belgium             41196.340
Sweden              38348.330
Name: Sales, dtype: float64


In [24]:
#---------مشتریان(VIP از نظر مبلغ خرید) VIP------------
#----------مشتریانی که بیشترین پول خرج کرده‌اند----------
# 280206 / 8911408 * 100 , بیشترین درآمد از مشتری حدود 3.1٪ کل فروش شرکت است.
top_customers = (
    df.groupby("CustomerID")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(top_customers)

CustomerID
14646.0    280206.02
18102.0    259657.30
17450.0    194550.79
16446.0    168472.50
14911.0    140450.72
12415.0    124914.53
14156.0    117379.63
17511.0     91062.38
16029.0     81024.84
12346.0     77183.60
Name: Sales, dtype: float64


In [25]:
#-----------مشتریانی که بیشترین سفارش ثبت کرده‌اند(مشتری وفادار)-----------
top_orders = (
    df.groupby("CustomerID")["InvoiceNo"]
      .nunique()
      .sort_values(ascending=False)
      .head(10)
)

print(top_orders)


CustomerID
12748.0    206
14911.0    199
17841.0    124
13089.0     97
14606.0     91
15311.0     91
12971.0     86
14646.0     73
16029.0     63
13408.0     62
Name: InvoiceNo, dtype: int64


In [26]:
#-----------مشتریانی که بیشترین تعداد کالا خریده‌اند(عمده فروش یا خرید حجمی)----------
top_quantity = (
    df.groupby("CustomerID")["Quantity"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(top_quantity)

CustomerID
14646.0    196915
16446.0     80997
14911.0     80263
12415.0     77374
12346.0     74215
17450.0     69993
17511.0     64549
18102.0     64124
13694.0     63312
14298.0     58343
Name: Quantity, dtype: int64


In [27]:
#-------------تحلیل کشورها-----------------
# 7276121 / 8911408 * 100,یعنی حدود 81٪ فروش.
country_sales = (
    df.groupby("Country")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

print(country_sales.head(10))

Country
United Kingdom    7276121.253
Netherlands        285446.340
EIRE               262171.560
Germany            226570.890
France             199531.680
Australia          138521.310
Spain               61577.110
Switzerland         56443.950
Belgium             41196.340
Sweden              38348.330
Name: Sales, dtype: float64


In [28]:
print(df["InvoiceDate"].max())

2011-12-09 12:50:00


In [29]:
#----------------تحلیل رفتار مشتری-------------
#---------------ساخت تاریخ مرجع(تاریخی که از ان به بعد رفتار مشتری را بررسی می کنیم)-------------------
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

print(snapshot_date)

2011-12-10 12:50:00


In [30]:
# ساخت جدول RFM برای هر مشتری
# Recency  : چند روز از آخرین خرید مشتری گذشته است
# Frequency: تعداد سفارش‌های ثبت‌شده توسط مشتری
# Monetary : مجموع مبلغ خرید مشتری

rfm = df.groupby("CustomerID").agg({

    # آخرین تاریخ خرید مشتری را پیدا می‌کند
    # سپس فاصله آن را تا تاریخ مرجع (snapshot_date) بر حسب روز محاسبه می‌کند
    "InvoiceDate": lambda x: (snapshot_date - x.max()).days,

    # تعداد فاکتورهای یکتای مشتری = تعداد سفارش‌ها
    "InvoiceNo": "nunique",

    # مجموع مبلغ خرید مشتری
    "Sales": "sum"

})

# تغییر نام ستون‌ها به نام‌های استاندارد RFM
rfm.columns = ["Recency", "Frequency", "Monetary"]

# نمایش 5 ردیف اول جدول RFM
print(rfm.head())

            Recency  Frequency  Monetary
CustomerID                              
12346.0         326          1  77183.60
12347.0           2          7   4310.00
12348.0          75          4   1797.24
12349.0          19          1   1757.55
12350.0         310          1    334.40


In [31]:
#------------امتیازدهی RFM----------------
# امتیازها بین 1 تا 5 هستند.هرچه بزرگ‌تر بهتر:
# 
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)

print(rfm.head())

            Recency  Frequency  Monetary R_Score F_Score M_Score
CustomerID                                                      
12346.0         326          1  77183.60       1       1       5
12347.0           2          7   4310.00       5       5       5
12348.0          75          4   1797.24       2       4       4
12349.0          19          1   1757.55       4       1       4
12350.0         310          1    334.40       1       1       2


In [32]:
#-------------RFM_Score--------------
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

print(rfm.head())

            Recency  Frequency  Monetary R_Score F_Score M_Score RFM_Score
CustomerID                                                                
12346.0         326          1  77183.60       1       1       5       115
12347.0           2          7   4310.00       5       5       5       555
12348.0          75          4   1797.24       2       4       4       244
12349.0          19          1   1757.55       4       1       4       414
12350.0         310          1    334.40       1       1       2       112


In [33]:
df.head(100000).to_csv("online_retail_sample.csv", index=False)

In [38]:
df.head(5000).to_csv("online_retail_10k.csv", index=False)